<a href="https://colab.research.google.com/github/wtryab-re/whats-your-eda-basics/blob/main/self_data_quality_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installation and Imports

In [242]:
!pip install -q numpy pandas

In [243]:
import numpy as np
import zipfile
import pandas as pd

In [244]:
#!/bin/bash
!kaggle datasets download mikhail1681/walmart-sales

Dataset URL: https://www.kaggle.com/datasets/mikhail1681/walmart-sales
License(s): other
walmart-sales.zip: Skipping, found more recently modified local copy (use --force to force download)


#Walmart Sales

#Unzip and Extract

In [245]:
fl = zipfile.ZipFile("walmart-sales.zip", "r")
fl.extractall()
fl.close()

##DataFrame

In [246]:
df = pd.read_csv("Walmart_Sales.csv")
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.57,211.10,8.11
1,1,12-02-2010,1641957.44,1,38.51,2.55,211.24,8.11
2,1,19-02-2010,1611968.17,0,39.93,2.51,211.29,8.11
3,1,26-02-2010,1409727.59,0,46.63,2.56,211.32,8.11
4,1,05-03-2010,1554806.68,0,46.50,2.62,211.35,8.11


Data Quality Checks

#Quick Checklist of things to do with the dataset:

####0. Overview
####1. Missing values summary
####2. Duplicates
####3. Data Type validation
####4. Constant and quasi constant columns
####5. ID like columns
####6. String inconsistencies
####7. High null columns
####8. High zero columns

##0. Overview

In [247]:
pd.set_option('display.max_columns', None)

In [248]:
pd.set_option("display.float_format", "{:.2f}".format)

In [249]:
df.shape

(6435, 8)

In [250]:
df.describe()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,6435.00,6435.00,6435.00,6435.00,6435.00,6435.00,6435.00
mean,23.00,1046964.88,0.07,60.66,3.36,171.58,8.00
std,12.99,564366.62,0.26,18.44,0.46,39.36,1.88
min,1.00,209986.25,0.00,-2.06,2.47,126.06,3.88
25%,12.00,553350.10,0.00,47.46,2.93,131.74,6.89
50%,23.00,960746.04,0.00,62.67,3.44,182.62,7.87
75%,34.00,1420158.66,0.00,74.94,3.73,212.74,8.62
max,45.00,3818686.45,1.00,100.14,4.47,227.23,14.31


In [251]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


In [252]:
df.dtypes

,0
Store,int64
Date,object
Weekly_Sales,float64
Holiday_Flag,int64
Temperature,float64
Fuel_Price,float64
CPI,float64
Unemployment,float64


In [253]:
df.columns

Index(['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature',
       'Fuel_Price', 'CPI', 'Unemployment'],
      dtype='object')

In [254]:
df.nunique()
#holiday is a category

,0
Store,45
Date,143
Weekly_Sales,6435
Holiday_Flag,2
Temperature,3528
Fuel_Price,892
CPI,2145
Unemployment,349


#1. Missing values summary

In [255]:
df.isna().sum()

,0
Store,0
Date,0
Weekly_Sales,0
Holiday_Flag,0
Temperature,0
Fuel_Price,0
CPI,0
Unemployment,0


#2. Duplicated

In [256]:
df.duplicated().sum()

np.int64(0)

In [257]:
df.duplicated().groupby(df.duplicated()).count()

,0
False,6435


#3. Data Type validation

In [258]:
df.dtypes

,0
Store,int64
Date,object
Weekly_Sales,float64
Holiday_Flag,int64
Temperature,float64
Fuel_Price,float64
CPI,float64
Unemployment,float64


#4. Constants and Quasi-Constants

In [259]:
df.nunique().sort_values()

,0
Holiday_Flag,2
Store,45
Date,143
Unemployment,349
Fuel_Price,892
CPI,2145
Temperature,3528
Weekly_Sales,6435


In [260]:
#Constant percentage check
n_rows=len(df)
nunique = df.nunique()
constant_cols = nunique[nunique==1].index.to_list()
print(constant_cols)

[]


In [261]:
#quasi constant check
quasi_constant_cols = []
for col in df.columns:
  top_freq= df[col].value_counts(normalize=True,dropna=False).values[0]
  if top_freq > 0.9 and col not in constant_cols:
    quasi_constant_cols.append(col)

quasi_constant_cols

['Holiday_Flag']

#5. ID like columns

In [262]:
for col in df.columns:
  if df[col].nunique() >= 0.95*len(df):
    print(col)

Weekly_Sales


#6. String Inconsistencies

In [263]:
df.dtypes

,0
Store,int64
Date,object
Weekly_Sales,float64
Holiday_Flag,int64
Temperature,float64
Fuel_Price,float64
CPI,float64
Unemployment,float64


In [264]:
object_cols = df.select_dtypes(include="object").columns.to_list()
object_cols

['Date']

In [265]:
df[object_cols] = df[object_cols].apply(lambda x: x.str.strip())

#7. High null columns

In [266]:
df.isna().sum()

,0
Store,0
Date,0
Weekly_Sales,0
Holiday_Flag,0
Temperature,0
Fuel_Price,0
CPI,0
Unemployment,0


#8. High Zero Columns

In [267]:
#find numeric columns
df.dtypes

,0
Store,int64
Date,object
Weekly_Sales,float64
Holiday_Flag,int64
Temperature,float64
Fuel_Price,float64
CPI,float64
Unemployment,float64


In [268]:
numeric_columns = df.select_dtypes(include=np.number).columns.to_list()
numeric_columns

['Store',
 'Weekly_Sales',
 'Holiday_Flag',
 'Temperature',
 'Fuel_Price',
 'CPI',
 'Unemployment']

In [269]:
high_zero_threshold = 40
zero_values = (df[numeric_columns]==0).mean()*100
zero_values.sort_values(ascending=False)

,0
Holiday_Flag,93.01
Store,0.00
Weekly_Sales,0.00
Temperature,0.00
Fuel_Price,0.00
CPI,0.00
Unemployment,0.00


In [270]:
zero_values[zero_values>high_zero_threshold]

,0
Holiday_Flag,93.01


#New Data Set -- Bike Sales

In [271]:
#!/bin/bash
!kaggle datasets download ratnarohith/uncleaned-bike-sales-data

Dataset URL: https://www.kaggle.com/datasets/ratnarohith/uncleaned-bike-sales-data
License(s): other
uncleaned-bike-sales-data.zip: Skipping, found more recently modified local copy (use --force to force download)


In [272]:
fil = zipfile.ZipFile("uncleaned-bike-sales-data.zip", "r")
fil.extractall()
fil.close()

#Quick Checklist of things to do with the dataset:

####0. Overview
####1. Missing values summary
####2. Duplicates
####3. Data Type validation
####4. Constant and quasi constant columns
####5. ID like columns
####6. String inconsistencies
####7. High null columns
####8. High zero columns

In [273]:
df = pd.read_excel("uncleaned bike sales data.xlsx")
df.head()

,Sales_Order #,Date,Day,Month,Year,Customer_Age,Age_Group,Customer_Gender,Country,State,Product_Category,Sub_Category,Product_Description,Order_Quantity,Unit_Cost,Unit_Price,Profit,Cost,Revenue
0,261695,2021-12-01,1.00,December,2021,39,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,"Mountain-200 Black, 46",4.00,1252,2295,4172,5008,9180
1,261695,2021-12-01,1.00,December,2021,44,Adults (35-64),M,United Kingdom,England,Bikes,Mountain Bikes,"Mountain-200 Silver, 42",1.00,1266,2320,1054,1266,2320
2,261697,2021-12-02,2.00,December,2021,37,Adults (35-64),M,United States,California,Bikes,Mountain Bikes,"Mountain-400-W Silver, 46",2.00,420,769,698,840,1538
3,261698,2021-12-02,2.00,December,2021,31,Young Adults (25-34),F,Australia,New South Wales,Bikes,Mountain Bikes,"Mountain-400-W Silver, 42",1.00,420,769,349,420,769
4,261699,2021-12-03,3.00,December,2021,37,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,"Mountain-200 Black, 46",2.00,0,2295,2086,0,4590


#0.Overview

In [274]:
df.shape

(89, 19)

In [275]:
df.columns

Index(['Sales_Order #', 'Date', 'Day', 'Month', 'Year', 'Customer_Age',
       'Age_Group', 'Customer_Gender', 'Country', 'State', 'Product_Category',
       'Sub_Category', 'Product_Description', 'Order_Quantity', ' Unit_Cost ',
       ' Unit_Price ', ' Profit ', ' Cost ', 'Revenue'],
      dtype='object')

In [276]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Sales_Order #        89 non-null     int64         
 1   Date                 89 non-null     datetime64[ns]
 2   Day                  88 non-null     float64       
 3   Month                89 non-null     object        
 4   Year                 89 non-null     int64         
 5   Customer_Age         89 non-null     int64         
 6   Age_Group            88 non-null     object        
 7   Customer_Gender      89 non-null     object        
 8   Country              89 non-null     object        
 9   State                89 non-null     object        
 10  Product_Category     89 non-null     object        
 11  Sub_Category         89 non-null     object        
 12  Product_Description  88 non-null     object        
 13  Order_Quantity       88 non-null     

In [277]:
df.dtypes

,0
Sales_Order #,int64
Date,datetime64[ns]
Day,float64
Month,object
Year,int64
Customer_Age,int64
Age_Group,object
Customer_Gender,object
Country,object
State,object


In [278]:
#to check for categorical stuff
df.nunique().sort_values()

,0
Year,1
Sub_Category,1
Product_Category,1
Month,2
Customer_Gender,2
Age_Group,3
Order_Quantity,4
Unit_Cost,8
Unit_Price,8
Country,9


#1. Missing Values

In [279]:
df.isna().sum().sort_values(ascending=False)

,0
Day,1
Product_Description,1
Age_Group,1
Order_Quantity,1
Sales_Order #,0
Date,0
Month,0
Customer_Gender,0
Country,0
Year,0


In [280]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_percentage = (df.isna().mean()*100).sort_values(ascending=False)

missing_df = pd.DataFrame({"missing_count":missing_count,"missing_percentage":missing_percentage})
missing_df[missing_df["missing_count"]>0]

,missing_count,missing_percentage
Day,1,1.12
Product_Description,1,1.12
Age_Group,1,1.12
Order_Quantity,1,1.12


#2. Duplicates

In [281]:
df.duplicated().sum()

np.int64(0)

In [282]:
df.duplicated().groupby(df.duplicated()).count()

,0
False,89


#3. Data Validation

In [283]:
df.dtypes

,0
Sales_Order #,int64
Date,datetime64[ns]
Day,float64
Month,object
Year,int64
Customer_Age,int64
Age_Group,object
Customer_Gender,object
Country,object
State,object


#4. Constant and quasi constant columns


In [284]:
constant_cols = df.nunique()[df.nunique()==1].index.to_list()
constant_cols

['Year', 'Product_Category', 'Sub_Category']

In [285]:
df[constant_cols].nunique()

,0
Year,1
Product_Category,1
Sub_Category,1


In [286]:
df.nunique().sort_values()

,0
Year,1
Sub_Category,1
Product_Category,1
Month,2
Customer_Gender,2
Age_Group,3
Order_Quantity,4
Unit_Cost,8
Unit_Price,8
Country,9


####quasiconstant

In [287]:
quasi_constant_cols = []
for col in df.columns:
  top_freq= df[col].value_counts(normalize=True,dropna=False).values[0]
  if top_freq > 0.9 and col not in constant_cols:
    quasi_constant_cols.append(col)

quasi_constant_cols

['Month']

#5. ID like columns

Cols that have almost the same amount of unique values as the length of the data

In [288]:
n_rows = len(df)
id_like_cols = []
for col in df.columns:
  if df[col].nunique() >= 0.95*n_rows:
    id_like_cols.append(col)

id_like_cols

['Sales_Order #']

In [289]:
df.shape

(89, 19)

#6. String inconsistencies
strip, lowercase

In [290]:
df.head(2)

,Sales_Order #,Date,Day,Month,Year,Customer_Age,Age_Group,Customer_Gender,Country,State,Product_Category,Sub_Category,Product_Description,Order_Quantity,Unit_Cost,Unit_Price,Profit,Cost,Revenue
0,261695,2021-12-01,1.00,December,2021,39,Adults (35-64),F,United States,California,Bikes,Mountain Bikes,"Mountain-200 Black, 46",4.00,1252,2295,4172,5008,9180
1,261695,2021-12-01,1.00,December,2021,44,Adults (35-64),M,United Kingdom,England,Bikes,Mountain Bikes,"Mountain-200 Silver, 42",1.00,1266,2320,1054,1266,2320


In [291]:
selected_datatypes = df.select_dtypes(include="object").columns.to_list()
selected_datatypes

['Month',
 'Age_Group',
 'Customer_Gender',
 'Country',
 'State',
 'Product_Category',
 'Sub_Category',
 'Product_Description']

In [292]:
for col in selected_datatypes:
  df[col] = df[col].str.strip()
  df[col] = df[col].str.lower()
df[selected_datatypes].head()

,Month,Age_Group,Customer_Gender,Country,State,Product_Category,Sub_Category,Product_Description
0,december,adults (35-64),f,united states,california,bikes,mountain bikes,"mountain-200 black, 46"
1,december,adults (35-64),m,united kingdom,england,bikes,mountain bikes,"mountain-200 silver, 42"
2,december,adults (35-64),m,united states,california,bikes,mountain bikes,"mountain-400-w silver, 46"
3,december,young adults (25-34),f,australia,new south wales,bikes,mountain bikes,"mountain-400-w silver, 42"
4,december,adults (35-64),f,united states,california,bikes,mountain bikes,"mountain-200 black, 46"


In [293]:
df["Month"]=df["Month"].replace("decmber","december")

#7. Null values check

In [294]:
df.head(2)

,Sales_Order #,Date,Day,Month,Year,Customer_Age,Age_Group,Customer_Gender,Country,State,Product_Category,Sub_Category,Product_Description,Order_Quantity,Unit_Cost,Unit_Price,Profit,Cost,Revenue
0,261695,2021-12-01,1.00,december,2021,39,adults (35-64),f,united states,california,bikes,mountain bikes,"mountain-200 black, 46",4.00,1252,2295,4172,5008,9180
1,261695,2021-12-01,1.00,december,2021,44,adults (35-64),m,united kingdom,england,bikes,mountain bikes,"mountain-200 silver, 42",1.00,1266,2320,1054,1266,2320


In [295]:
missing_count = df.isna().sum().sort_values(ascending=False)
missing_percentage = df.isna().mean()*100
missing_df = pd.DataFrame({"missing_count":missing_count,"missing_percentage":missing_percentage})
missing_df[missing_df["missing_count"]>0]

,missing_count,missing_percentage
Age_Group,1,1.12
Day,1,1.12
Order_Quantity,1,1.12
Product_Description,1,1.12


#Zero Value Check

In [296]:
numeric_columns = df.select_dtypes(include=np.number).columns.to_list()
#numeric_columns.remove("survived")
numeric_columns

['Sales_Order #',
 'Day',
 'Year',
 'Customer_Age',
 'Order_Quantity',
 ' Unit_Cost ',
 ' Unit_Price ',
 ' Profit ',
 ' Cost ',
 'Revenue']

In [305]:
zero_values = (df[numeric_columns]==0).mean()*100
zero_values.sort_values(ascending=False)

,0
Revenue,2.25
Cost,2.25
Unit_Cost,1.12
Unit_Price,1.12
Day,0.00
Sales_Order #,0.00
Order_Quantity,0.00
Customer_Age,0.00
Year,0.00
Profit,0.00
